In [14]:
import os

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

import json
from tqdm import tqdm
os.environ["CUDA_VISIBLE_DEVICES"] = "0"  

from datetime import datetime
from dataclasses import asdict, dataclass, field
from sklearn.model_selection import train_test_split
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.utils as vutils
from torch.utils.data import DataLoader

from sklearn.model_selection import train_test_split

from datasets import FastDatasetDIDC
from gan_basic import DiscriminatorModel
from unet_advanced import UNetAdvanced as GeneratorModel
from mt_DIDC_config import GROUPING_RULES, LABEL2LABEL, PROPERTY_KEY, NEW_LABELS
from utils import setup_logger, set_reproducibility

from tx_GAN_train import GANTrainerConfig, GeneratorConfig, BSSFPConfig, DiscrConfig, CustomDatasetTexturizer, bSSFPSimulator

%load_ext autoreload
%autoreload 2


The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [12]:

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Selected device: {device}")
print(f'Num available GPUs: ', torch.cuda.device_count())


p = torch.cuda.get_device_properties()
print(f"Device: {p.name} (Memory: {p.total_memory / 1e9:.2f} GB)")

Selected device: cuda
Num available GPUs:  1
Device: NVIDIA TITAN RTX (Memory: 25.19 GB)


In [13]:
DATA_DIR = "./DIDC_multiclass_coro_v2_prep_2"

In [6]:
class TexturizerGANInferencer:
    def __init__(self, config: GANTrainerConfig, gen_model: nn.Module, bssfp_sim: nn.Module, device="cuda"):
        self.config = config
        self.device = device
        
        self.gen = gen_model.to(self.device).eval()
        self.bssfp_sim = bssfp_sim.to(self.device).eval()

    @torch.no_grad()
    def generate(self, batch):
        """Esegue il forward pass del Generatore e del Simulatore bSSFP."""
        # 1. Preparazione della maschera originale (1 canale)
        original_mask = batch['input_label'].unsqueeze(1).float().to(self.device)
        
        # 2. Codifica One-Hot (esattamente come nel train_step)
        condition = original_mask.clone()
        if self.config.gen_model.in_ch > 1:
            condition = F.one_hot(
                original_mask.squeeze(1).long(), 
                num_classes=self.config.gen_model.in_ch
            ).permute(0, 3, 1, 2).float().contiguous()

        # 3. Generazione degli offset (T1, T2, PD)
        pred_offsets = self.gen(condition)
        
        # 4. Simulazione Fisica (usando la maschera a 1 canale!)
        pred_img = self.bssfp_sim(pred_offsets, original_mask)
        if pred_img.ndim == 3:
            pred_img = pred_img.unsqueeze(1)
            
        return {
            'pred_offsets': pred_offsets.cpu(),
            'pred_img': pred_img.cpu()
        }

In [ ]:
import matplotlib.patches as mpatches

def evaluate_GAN(dir_name, device="cuda"):
    print(f"Loading experiment from: {dir_name}")

    # 1. Caricamento robusto delle configurazioni annidate
    with open(os.path.join(dir_name, "training_config.json"), "r") as f:
        config_dict = json.load(f)
        
        # Ricostruiamo le dataclass annidate
        gen_cfg = GeneratorConfig(**config_dict.get('gen_model', {}))
        bssfp_cfg = BSSFPConfig(**config_dict.get('bssfp_model', {}))
        discr_cfg = DiscrConfig(**config_dict.get('discr_model', {}))
        
        training_config = GANTrainerConfig(
            **{k: v for k, v in config_dict.items() if hasattr(GANTrainerConfig, k) and k not in ['gen_model', 'bssfp_model', 'discr_model']}
        )
        training_config.gen_model = gen_cfg
        training_config.bssfp_model = bssfp_cfg
        training_config.discr_model = discr_cfg

    # Riproducibilità rigorosa
    set_reproducibility(training_config.seed)

    # 2. Caricamento liste e label
    with open(os.path.join(dir_name, "train_val_split.json"), "r") as f:
        train_val_split = json.load(f)

    with open(os.path.join(dir_name, "grouping_rules_and_labels.json"), "r") as f:
        new_labels = json.load(f)['new_labels']

    # 3. Preparazione Dataloader (e prelievo del batch CENTRALE)
    val_dataset = CustomDatasetTexturizer(training_config.data_path, file_list=train_val_split['val_indices'])
    dataloader = DataLoader(val_dataset, batch_size=training_config.batch_size_per_gpu, shuffle=False)
    
    target_batch_idx = len(dataloader) // 2
    batch = None
    for i, b in enumerate(dataloader):
        if i == target_batch_idx:
            batch = b
            break
            
    print(f"Selected validation batch shape: {batch['input_label'].shape}")

    # 4. Inizializzazione Modelli e caricamento Pesi
    checkpoint_path = os.path.join(dir_name, "checkpoint")
    print(f"Loading Models from {checkpoint_path}...")

    model = GeneratorModel(**asdict(training_config.gen_model))
    gen_weights = torch.load(os.path.join(checkpoint_path, "generator.pth"), map_location=device)
    model.load_state_dict(gen_weights)

    bssfp_sim = bSSFPSimulator(training_config.bssfp_model)

    # 5. Inferenza
    inferencer = TexturizerGANInferencer(
        config=training_config,
        gen_model=model,
        bssfp_sim=bssfp_sim,
        device=device
    )

    outputs = inferencer.generate(batch)
    generated_mri = outputs['pred_img'].numpy()
    
    # 6. Plotting delle Metriche (se presenti)
    metrics_path = os.path.join(dir_name, "metrics_history.json")
    if os.path.exists(metrics_path):
        with open(metrics_path, "r") as f:
            metrics_history = json.load(f)
            
        plot_metrics = {k: v for k, v in metrics_history.items() if 'loss' in k.lower()}
        if plot_metrics:
            fig, ax = plt.subplots(1, len(plot_metrics), figsize=(15, 5), squeeze=False)
            for i, (metric_name, metric_values) in enumerate(plot_metrics.items()):
                if not metric_values: continue
                ax[0, i].plot(metric_values, linewidth=2, color='royalblue' if 'loss' in metric_name else 'darkorange')
                ax[0, i].set_title(metric_name.replace('_', ' ').title(), fontweight='bold')
                ax[0, i].set_xlabel('Epoch / Step')
                ax[0, i].set_ylabel("Value")
                ax[0, i].grid(linestyle='--', alpha=0.7)
            plt.tight_layout()
            plt.show()

    # 7. Visualizzazione Anatomica vs MRI Generata
    actual_batch_size = batch['input_label'].shape[0]
    num_classes = len(new_labels)

    cond_visual = batch['input_label'].numpy()
    gt_mri = batch['mri_slice'].numpy()

    cmap_gt = plt.get_cmap('nipy_spectral', num_classes)
    
    # Riduciamo il plot a un massimo di 8 immagini per non impallare matplotlib
    n_plot = min(actual_batch_size, 8)
    fig, ax = plt.subplots(n_plot, 3, figsize=(12, 4 * n_plot), squeeze=False)

    for i in range(n_plot):
        # Condizione (Label Map Discreta)
        ax[i, 0].imshow(cond_visual[i], cmap=cmap_gt, vmin=0, vmax=num_classes-1) 
        ax[i, 0].set_title('Condition Map', fontweight='bold')
        ax[i, 0].axis('off')

        # Real MRI (Scala di grigi)
        ax[i, 1].imshow(gt_mri[i], cmap='gray')
        ax[i, 1].set_title('Ground Truth MRI', fontweight='bold')
        ax[i, 1].axis('off')

        # Fake MRI (Scala di grigi) - Facciamo lo squeeze per togliere la dim canale
        ax[i, 2].imshow(generated_mri[i].squeeze(0), cmap='gray')
        ax[i, 2].set_title('bSSFP Generated', fontweight='bold')
        ax[i, 2].axis('off')

    # Creazione Legenda
    legend_patches = [mpatches.Patch(color=cmap_gt(c), label=label) for c, label in enumerate(new_labels)]
    plt.tight_layout(rect=[0, 0, 0.85, 1]) 
    fig.legend(handles=legend_patches, loc='center right', bbox_to_anchor=(1.08, 0.5), title="Tissue Classes")

    plt.show()